# DFI Video Maker — batch renderer (Sheet + Drive)

Renders a spinning-record video for every `Render? = TRUE` row in the DFI track
sheet: cover art spinning like a record (with motion blur), the DFI branding
overlaid, and a snippet of the track underneath. Finished MP4s are saved to a
Google Drive folder.

**How to run:** check the **Config** cell below, then **Runtime → Run all** and
sign in with the shared Google account when prompted. Everything else is automatic.


## 1 · Config  ← check these

In [ ]:
# ---- IDs (already filled in for the DFI account) ---------------------------
SHEET_ID               = "1S_MFhIt0V8OJWZ8IMYAFf9_bQJ2WVMcJpYLe5uCqlRY"
WORKSHEET_NAME         = None        # None = first tab; or a tab name (str) / index (int)
DRIVE_OUTPUT_FOLDER_ID = "1iW4LFcTWxxA2qze4jga0O7s6EGG5c8cw"   # finished videos land here

# ---- Artwork QA ------------------------------------------------------------
# True  = look, don't render: shows every track's artwork in one grid so you can
#         spot wrong or junk covers (common with unknown rips) BEFORE rendering.
# False = render normally.
PREVIEW_ONLY           = False

# True  = after showing the artwork, ask y/n before rendering (the run WAITS for
#         your answer). False = render straight away without asking.
CONFIRM_BEFORE_RENDER  = True

# How many tracks to download from Drive at the same time. Downloads are the slow
# part of a big batch; 4 is a sensible default. Set to 1 to download one by one.
DOWNLOAD_WORKERS       = 4

# ---- Sheet column headers (edit here if you rename a column in the sheet) ---
# These must match the header text in your sheet EXACTLY (spelling, spaces, case).
COL_TRACK      = "Track"
COL_ARTIST     = "Artist"
COL_AUDIO      = "Drive audio file"
COL_ARTWORK    = "Drive artwork file*"
COL_CLIP_START = "Clip start"
COL_RENDER     = "Render?"

# ---- Branding overlay (optional) -------------------------------------------
# A transparent PNG the SAME size as the canvas, kept in Google Drive.
# Paste its share link below. To change the branding, paste a different link.
# Leave it as "" for no overlay.
OVERLAY_DRIVE_LINK     = "https://drive.google.com/file/d/12ehRHJIcCrqb2_jgmjEiToewc4KL28hu/view"  # paste a different link to change branding

# ---- Fallback artwork (optional) -------------------------------------------
# Used as the record for any track with no embedded cover art and no override.
# A square image kept in Google Drive — paste its share link below.
# Leave it as "" to skip those rows instead.
FALLBACK_DRIVE_LINK    = "https://drive.google.com/file/d/1f7HqsT1bpu2rkGcJRJkmf1dvVhKBCFq8/view"

# ---- Brand font (for the burnt-in track/artist caption) ---------------------
# The .otf/.ttf kept in Google Drive. Without it Colab falls back to a plain
# default font, so the caption won't look on-brand. Leave "" to use a default.
FONT_DRIVE_LINK        = "https://drive.google.com/file/d/10Ey-lZQRHi09BswDOyQPRqHocUOHRv90/view"

# ---- Look & feel -----------------------------------------------------------
CLIP_LENGTH_SECONDS = 25             # clip length for every row
SPIN_PERIOD_SECONDS = 6              # seconds per full rotation
FPS                 = 30
CANVAS_W            = 1080           # 1080 x 1350 = 4:5 (Instagram grid)
CANVAS_H            = 1350           # set 1080 for square (use a matching overlay!)
CIRCLE_DIAMETER     = 830            # diameter of the spinning record
DISC_OFFSET_Y       = 0              # nudge the record up (-) / down (+) from centre
BG_COLOUR           = "black"
MOTION_BLUR_SAMPLES = 10             # 1 = no blur
SHUTTER_FRACTION    = 0.7            # blur amount (0.5 = 180-degree shutter)

## 2 · Install dependencies
(ffmpeg via apt; the Python libs are already in Colab but we pin them to be safe.)

In [ ]:
import importlib, shutil, subprocess, sys, time

_t0 = time.time()

# Colab already ships most of this — only install what's genuinely missing,
# which saves ~30-60s of startup on every fresh runtime.
_needed = [("pillow", "PIL"), ("mutagen", "mutagen"), ("numpy", "numpy"),
           ("gspread", "gspread"), ("google-api-python-client", "googleapiclient"),
           ("google-auth-httplib2", "google_auth_httplib2")]


def _installed(module):
    try:
        importlib.import_module(module)
        return True
    except ImportError:
        return False


missing = [pkg for pkg, module in _needed if not _installed(module)]
if missing:
    print(f"Installing: {', '.join(missing)} ...")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", *missing], check=True)
else:
    print("Python packages: already present.")

if shutil.which("ffmpeg") and shutil.which("ffprobe"):
    print("ffmpeg: already installed.")
else:
    print("Installing ffmpeg ...")
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print(f"Dependencies ready in {time.time() - _t0:.1f}s.")


## 3 · Sign in to Google
Authorises **you** for Sheets + Drive. Use the shared DFI account when the pop-up asks.

In [ ]:
from google.colab import auth
auth.authenticate_user()

import gspread
from google.auth import default
from googleapiclient.discovery import build

creds, _ = default()
gc = gspread.authorize(creds)
drive = build("drive", "v3")
print("Signed in — connected to Google Sheets + Drive.")

## 4 · Load the render engine
Downloads the latest engine from GitHub and imports it, so you always run the newest version — no need to re-upload the notebook when the engine improves.

In [ ]:
# Fetch the latest engine from GitHub (this is what makes the notebook a thin
# "launcher": the engine lives in the repo, so improvements arrive automatically).
import urllib.request
ENGINE_URL = "https://raw.githubusercontent.com/tiredlabrador/dfi-video-maker/main/generate_video.py"
urllib.request.urlretrieve(ENGINE_URL, "generate_video.py")
print("Engine downloaded from GitHub.")

In [ ]:
import importlib, generate_video
importlib.reload(generate_video)
from generate_video import (RenderConfig, render_video,
                            NoArtworkError, RenderError, sanitise_filename,
                            resolve_artwork, make_contact_sheet,
                            preflight, format_preflight)

CFG = RenderConfig(
    clip_length_seconds=CLIP_LENGTH_SECONDS,
    spin_period_seconds=SPIN_PERIOD_SECONDS,
    fps=FPS, canvas_w=CANVAS_W, canvas_h=CANVAS_H,
    circle_diameter=CIRCLE_DIAMETER, disc_offset_y=DISC_OFFSET_Y,
    bg_colour=BG_COLOUR,
    motion_blur_samples=MOTION_BLUR_SAMPLES, shutter_fraction=SHUTTER_FRACTION,
)
print("Engine loaded.")

## 5 · Drive / URL helpers
Resolve a Drive share link (or a plain image URL) to a downloaded file, and upload results back to Drive.

In [ ]:
import io, os, re, tempfile, requests
from urllib.parse import urlparse
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload

def extract_drive_id(link):
    """Pull a Drive file id out of the common share-link shapes (or a bare id)."""
    link = str(link).strip()
    for pat in (r"/d/([A-Za-z0-9_-]{20,})", r"[?&]id=([A-Za-z0-9_-]{20,})"):
        m = re.search(pat, link)
        if m:
            return m.group(1)
    if re.fullmatch(r"[A-Za-z0-9_-]{20,}", link):
        return link
    return None

def _drive_name(file_id, drive_client=None):
    meta = (drive_client or drive).files().get(fileId=file_id, fields="name",
                             supportsAllDrives=True).execute()
    return meta.get("name", "")

def _download_drive(file_id, dest, drive_client=None):
    req = (drive_client or drive).files().get_media(fileId=file_id, supportsAllDrives=True)
    with io.FileIO(dest, "wb") as fh:
        downloader = MediaIoBaseDownload(fh, req)
        done = False
        while not done:
            _, done = downloader.next_chunk()
    return dest

def fetch_source(link, dest_base, drive_client=None):
    """
    Download `link` to `dest_base` + an inferred extension; return the path.
    Handles Drive share links, bare Drive ids, and plain http(s) image URLs.
    """
    link = str(link).strip()
    is_drive = ("drive.google.com" in link) or (not link.lower().startswith("http"))
    if is_drive:
        file_id = extract_drive_id(link)
        if not file_id:
            raise ValueError(f"Could not parse a Drive id from: {link!r}")
        ext = os.path.splitext(_drive_name(file_id, drive_client))[1]
        return _download_drive(file_id, dest_base + ext, drive_client)
    ext = os.path.splitext(urlparse(link).path)[1]
    dest = dest_base + ext
    r = requests.get(link, timeout=60)
    r.raise_for_status()
    with open(dest, "wb") as fh:
        fh.write(r.content)
    return dest

def upload_to_drive(local_path, name, folder_id):
    meta = {"name": name, "parents": [folder_id]}
    media = MediaFileUpload(local_path, mimetype="video/mp4", resumable=True)
    return drive.files().create(body=meta, media_body=media,
                                fields="id,webViewLink",
                                supportsAllDrives=True).execute()

def get_or_create_folder(name, parent_id):
    """Return the id of a subfolder `name` inside `parent_id`, creating it if
    it doesn't exist. Re-running a batch reuses the same folder (no duplicates)."""
    safe = name.replace("\\", "\\\\").replace("'", "\\'")
    q = ("name = '" + safe + "' and mimeType = 'application/vnd.google-apps.folder' "
         "and '" + parent_id + "' in parents and trashed = false")
    hits = drive.files().list(q=q, spaces="drive", fields="files(id, name)",
                              supportsAllDrives=True,
                              includeItemsFromAllDrives=True).execute().get("files", [])
    if hits:
        return hits[0]["id"]
    meta = {"name": name, "mimeType": "application/vnd.google-apps.folder",
            "parents": [parent_id]}
    return drive.files().create(body=meta, fields="id",
                                supportsAllDrives=True).execute()["id"]

import threading
from concurrent.futures import ThreadPoolExecutor

_thread_local = threading.local()

def _drive_for_thread():
    """
    googleapiclient clients are NOT thread-safe, so give each worker thread its
    own. Without this, parallel downloads corrupt each other's responses.
    """
    client = getattr(_thread_local, "drive", None)
    if client is None:
        client = build("drive", "v3", credentials=creds)
        _thread_local.drive = client
    return client

def fetch_many(items, workers=4):
    """
    items: list of (key, link, dest_base). Downloads them concurrently and returns
    {key: path} / {key: Exception}. Order of the input is irrelevant — callers
    look results up by key so logging stays deterministic.
    """
    results = {}

    def one(item):
        key, link, dest_base = item
        try:
            return key, fetch_source(link, dest_base, drive_client=_drive_for_thread())
        except Exception as exc:
            return key, exc

    if workers <= 1 or len(items) <= 1:
        for item in items:
            key, value = one(item)
            results[key] = value
        return results

    with ThreadPoolExecutor(max_workers=workers) as pool:
        for key, value in pool.map(one, items):
            results[key] = value
    return results

print("Helpers ready.")

## 6 · Read the track sheet

In [ ]:
sh = gc.open_by_key(SHEET_ID)
if WORKSHEET_NAME in (None, ""):
    ws = sh.sheet1
elif isinstance(WORKSHEET_NAME, int):
    ws = sh.get_worksheet(WORKSHEET_NAME)
else:
    ws = sh.worksheet(WORKSHEET_NAME)

rows = ws.get_all_records()   # list of dicts keyed by the header row
print(f"Read {len(rows)} data rows from tab '{ws.title}'.")

## 7 · Batch render
Downloads the branding overlay once, then renders every `Render? = TRUE` row. One bad row never halts the batch — each is caught, logged, and the run continues. A summary prints at the end.

In [ ]:
# --- assets: download once from Drive ---------------------------------------
CFG.overlay_path = None
if OVERLAY_DRIVE_LINK.strip():
    try:
        _ov_dir = tempfile.mkdtemp()
        CFG.overlay_path = fetch_source(OVERLAY_DRIVE_LINK,
                                        os.path.join(_ov_dir, "overlay"))
        print("Branding overlay loaded from Drive.")
    except Exception as exc:
        print(f"WARNING: could not load overlay ({exc}). Rendering without branding.")
        CFG.overlay_path = None

CFG.fallback_path = None
if FALLBACK_DRIVE_LINK.strip():
    try:
        _fb_dir = tempfile.mkdtemp()
        CFG.fallback_path = fetch_source(FALLBACK_DRIVE_LINK,
                                         os.path.join(_fb_dir, "fallback"))
        print("Fallback artwork loaded from Drive.")
    except Exception as exc:
        print(f"WARNING: could not load fallback artwork ({exc}). "
              f"Rows with no artwork will be skipped.")
        CFG.fallback_path = None

if FONT_DRIVE_LINK.strip():
    try:
        _ft_dir = tempfile.mkdtemp()
        CFG.font_path = fetch_source(FONT_DRIVE_LINK, os.path.join(_ft_dir, "font"))
        print("Brand font loaded from Drive.")
    except Exception as exc:
        print(f"WARNING: could not load brand font ({exc}). Using a default font.")
        CFG.font_path = None


def is_true(v):
    return v is True or str(v).strip().upper() in ("TRUE", "YES", "1")


def ask_yes_no(question):
    """Ask in the Colab output. Anything but 'y' means stop — the safe default."""
    try:
        answer = input(f"{question} [y/N]: ").strip().lower()
    except (EOFError, OSError):
        print("No input available (not an interactive session) — stopping to be safe.")
        return False
    return answer in ("y", "yes")


flagged = [(i, row) for i, row in enumerate(rows, start=2)
           if is_true(row.get(COL_RENDER))]

# --- 1. PRE-FLIGHT: check every flagged row before downloading anything -------
report = preflight([{
    "row": i,
    "track": row.get(COL_TRACK, ""),
    "artist": row.get(COL_ARTIST, ""),
    "audio_link": row.get(COL_AUDIO, ""),
    "clip_start": row.get(COL_CLIP_START, ""),
} for i, row in flagged])
print(f"\nChecking {len(flagged)} flagged row(s) on tab {ws.title!r} ...")
print(format_preflight(report))

rendered, skipped, failed, previews, jobs = [], [], [], [], []
batch_folder_id = None

if not report["ok"]:
    print("Fix the errors above, then Run all again.")
else:
    with tempfile.TemporaryDirectory() as work:
        # --- 2. FETCH each track and work out which artwork it will use -------
        downloads = []
        for i, row in flagged:
            label = (f"row {i}: {str(row.get(COL_ARTIST, '')).strip()} - "
                     f"{str(row.get(COL_TRACK, '')).strip()}").strip(" -")
            audio_link = str(row.get(COL_AUDIO, "")).strip()
            if not audio_link:
                skipped.append((label, "Audio file blank"))
                print(f"SKIP  {label} — Audio file blank")
                continue
            downloads.append((("audio", i), audio_link, os.path.join(work, f"audio_{i}")))
            artwork_link = str(row.get(COL_ARTWORK, "")).strip()
            if artwork_link:
                downloads.append((("art", i), artwork_link, os.path.join(work, f"art_{i}")))

        if downloads:
            print(f"\nDownloading {len(downloads)} file(s) "
                  f"({DOWNLOAD_WORKERS} at a time) ...")
        fetched = fetch_many(downloads, workers=DOWNLOAD_WORKERS)

        for i, row in flagged:
            track = str(row.get(COL_TRACK, "")).strip()
            artist = str(row.get(COL_ARTIST, "")).strip()
            label = f"row {i}: {artist} - {track}".strip(" -")

            if ("audio", i) not in fetched:
                continue                      # already logged as skipped above

            try:
                audio_path = fetched[("audio", i)]
                if isinstance(audio_path, Exception):
                    raise audio_path

                artwork_path = fetched.get(("art", i))
                if isinstance(artwork_path, Exception):
                    raise artwork_path

                art_img, art_src = resolve_artwork(audio_path, artwork_path, CFG,
                                                   track=track, artist=artist)
                previews.append((art_img, f"{artist} - {track}"[:44] +
                                 f"\n[{art_src}] {art_img.width}x{art_img.height}"))
                jobs.append({"row": i, "track": track, "artist": artist, "label": label,
                             "audio_path": audio_path, "artwork_path": artwork_path,
                             "clip_start": row.get(COL_CLIP_START, "0:00")})
                print(f"READY {label} — artwork: {art_src} "
                      f"({art_img.width}x{art_img.height})")

            except NoArtworkError:
                skipped.append((label, "no artwork (no override and no embedded art)"))
                print(f"SKIP  {label} — no artwork")
            except Exception as exc:
                failed.append((label, str(exc).splitlines()[0] if str(exc) else repr(exc)))
                print(f"FAIL  {label} — {exc}")

        # --- 3. SHOW the artwork, then ask whether to go ahead ----------------
        proceed = bool(jobs)
        if previews:
            from IPython.display import display
            print("\n" + "=" * 64)
            print(f"  ARTWORK CHECK — {len(previews)} cover(s)")
            print("  Anything wrong? Answer 'n', then put an image link in the")
            print(f"  '{COL_ARTWORK}' column of the sheet and Run all again.")
            print("=" * 64)
            display(make_contact_sheet(previews, cols=3, thumb=300, cfg=CFG))

        if PREVIEW_ONLY:
            proceed = False
            print("\nPREVIEW_ONLY is on — stopping here. Nothing was rendered.")
        elif proceed and CONFIRM_BEFORE_RENDER:
            proceed = ask_yes_no(f"\nRender these {len(jobs)} video(s)?")
            if not proceed:
                print("Stopped. Nothing was rendered.")

        # --- 4. RENDER the approved tracks ------------------------------------
        if proceed:
            for job in jobs:
                try:
                    name = sanitise_filename(f"{job['artist']} - {job['track']}") + ".mp4"
                    out_path = os.path.join(work, name)
                    render_video(job["audio_path"], job["artwork_path"],
                                 job["clip_start"], out_path, CFG,
                                 track=job["track"], artist=job["artist"])

                    if batch_folder_id is None:
                        batch_folder_id = get_or_create_folder(ws.title,
                                                               DRIVE_OUTPUT_FOLDER_ID)
                        print(f"Saving videos to output subfolder: {ws.title!r}")
                    info = upload_to_drive(out_path, name, batch_folder_id)
                    rendered.append((job["label"], name, info.get("webViewLink", "")))
                    print(f"OK    {job['label']} -> {name}")
                except Exception as exc:
                    failed.append((job["label"],
                                   str(exc).splitlines()[0] if str(exc) else repr(exc)))
                    print(f"FAIL  {job['label']} — {exc}")

print("\n" + "=" * 64)
print("  BATCH SUMMARY")
print("=" * 64)
print(f"  Rendered OK : {len(rendered)}")
for label, name, link in rendered:
    print(f"      - {name}   {link}")
print(f"  Skipped     : {len(skipped)}")
for label, reason in skipped:
    print(f"      - {label}  ({reason})")
print(f"  Failed      : {len(failed)}")
for label, reason in failed:
    print(f"      - {label}  ({reason})")
print("=" * 64)
